# Credit Risk Pipeline — SDD Specification & Source Inspection

Spec-Driven Development deliverable for the Bronze → Silver → Gold medallion pipeline over the Kaggle Credit Risk Dataset. This notebook contains the **verified source-data inspection** and the **technical specification for review — no implementation code until the spec is approved**.

In [0]:
%sql
-- Step 1a: Verify the actual schema of the Bronze raw table
DESCRIBE TABLE workspace.bronze.credit_risk_raw

In [0]:
%sql
-- Step 1b: Row count and missing values per column
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN person_age IS NULL THEN 1 ELSE 0 END)                AS nulls_person_age,
  SUM(CASE WHEN person_income IS NULL THEN 1 ELSE 0 END)            AS nulls_person_income,
  SUM(CASE WHEN person_home_ownership IS NULL THEN 1 ELSE 0 END)   AS nulls_person_home_ownership,
  SUM(CASE WHEN person_emp_length IS NULL THEN 1 ELSE 0 END)        AS nulls_person_emp_length,
  SUM(CASE WHEN loan_intent IS NULL THEN 1 ELSE 0 END)              AS nulls_loan_intent,
  SUM(CASE WHEN loan_grade IS NULL THEN 1 ELSE 0 END)               AS nulls_loan_grade,
  SUM(CASE WHEN loan_amnt IS NULL THEN 1 ELSE 0 END)                AS nulls_loan_amnt,
  SUM(CASE WHEN loan_int_rate IS NULL THEN 1 ELSE 0 END)            AS nulls_loan_int_rate,
  SUM(CASE WHEN loan_status IS NULL THEN 1 ELSE 0 END)              AS nulls_loan_status,
  SUM(CASE WHEN loan_percent_income IS NULL THEN 1 ELSE 0 END)      AS nulls_loan_percent_income,
  SUM(CASE WHEN cb_person_default_on_file IS NULL THEN 1 ELSE 0 END) AS nulls_cb_person_default_on_file,
  SUM(CASE WHEN cb_person_cred_hist_length IS NULL THEN 1 ELSE 0 END) AS nulls_cb_person_cred_hist_length
FROM workspace.bronze.credit_risk_raw

In [0]:
%sql
-- Step 1c: Basic statistics and percentiles for all numeric columns
SELECT 'person_age' AS column_name, MIN(person_age) AS min, PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY person_age) AS p05, PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY person_age) AS p25, PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY person_age) AS median, PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY person_age) AS p95, MAX(person_age) AS max, ROUND(AVG(person_age), 2) AS mean FROM workspace.bronze.credit_risk_raw
UNION ALL
SELECT 'person_income', MIN(person_income), PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY person_income), PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY person_income), PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY person_income), PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY person_income), MAX(person_income), ROUND(AVG(person_income), 2) FROM workspace.bronze.credit_risk_raw
UNION ALL
SELECT 'person_emp_length', MIN(person_emp_length), PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY person_emp_length), PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY person_emp_length), PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY person_emp_length), PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY person_emp_length), MAX(person_emp_length), ROUND(AVG(person_emp_length), 2) FROM workspace.bronze.credit_risk_raw
UNION ALL
SELECT 'loan_amnt', MIN(loan_amnt), PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY loan_amnt), PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY loan_amnt), PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY loan_amnt), PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY loan_amnt), MAX(loan_amnt), ROUND(AVG(loan_amnt), 2) FROM workspace.bronze.credit_risk_raw
UNION ALL
SELECT 'loan_int_rate', MIN(loan_int_rate), PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY loan_int_rate), PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY loan_int_rate), PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY loan_int_rate), PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY loan_int_rate), MAX(loan_int_rate), ROUND(AVG(loan_int_rate), 2) FROM workspace.bronze.credit_risk_raw
UNION ALL
SELECT 'loan_percent_income', MIN(loan_percent_income), PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY loan_percent_income), PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY loan_percent_income), PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY loan_percent_income), PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY loan_percent_income), MAX(loan_percent_income), ROUND(AVG(loan_percent_income), 4) FROM workspace.bronze.credit_risk_raw
UNION ALL
SELECT 'cb_person_cred_hist_length', MIN(cb_person_cred_hist_length), PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY cb_person_cred_hist_length), PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY cb_person_cred_hist_length), PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY cb_person_cred_hist_length), PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY cb_person_cred_hist_length), MAX(cb_person_cred_hist_length), ROUND(AVG(cb_person_cred_hist_length), 2) FROM workspace.bronze.credit_risk_raw

In [0]:
%sql
-- Step 1d: Enumerate all values of every categorical column (no assumed values)
SELECT 'loan_status' AS column_name, CAST(loan_status AS STRING) AS value, COUNT(*) AS n FROM workspace.bronze.credit_risk_raw GROUP BY loan_status
UNION ALL
SELECT 'person_home_ownership', person_home_ownership, COUNT(*) FROM workspace.bronze.credit_risk_raw GROUP BY person_home_ownership
UNION ALL
SELECT 'loan_intent', loan_intent, COUNT(*) FROM workspace.bronze.credit_risk_raw GROUP BY loan_intent
UNION ALL
SELECT 'loan_grade', loan_grade, COUNT(*) FROM workspace.bronze.credit_risk_raw GROUP BY loan_grade
UNION ALL
SELECT 'cb_person_default_on_file', cb_person_default_on_file, COUNT(*) FROM workspace.bronze.credit_risk_raw GROUP BY cb_person_default_on_file
ORDER BY column_name, n DESC

In [0]:
%sql
-- Step 1e: Exact-duplicate check over all 12 source columns
WITH counts AS (
  SELECT COUNT(*) AS total_rows FROM workspace.bronze.credit_risk_raw
),
distinct_rows AS (
  SELECT COUNT(*) AS distinct_rows FROM (
    SELECT 1 FROM workspace.bronze.credit_risk_raw
    GROUP BY person_age, person_income, person_home_ownership, person_emp_length,
             loan_intent, loan_grade, loan_amnt, loan_int_rate, loan_status,
             loan_percent_income, cb_person_default_on_file, cb_person_cred_hist_length
  )
)
SELECT total_rows, distinct_rows, total_rows - distinct_rows AS duplicate_rows
FROM counts, distinct_rows

In [0]:
%sql
-- Step 1f: Data-quality anomalies and logical-consistency checks (counts only)
SELECT
  SUM(CASE WHEN person_age > 100 THEN 1 ELSE 0 END)                              AS age_gt_100,
  SUM(CASE WHEN person_age > 90 AND person_age <= 100 THEN 1 ELSE 0 END)        AS age_91_to_100,
  SUM(CASE WHEN person_emp_length > person_age THEN 1 ELSE 0 END)              AS emp_length_gt_age,
  SUM(CASE WHEN person_income >= 1000000 THEN 1 ELSE 0 END)                     AS income_gte_1m,
  SUM(CASE WHEN loan_int_rate = 0 THEN 1 ELSE 0 END)                            AS int_rate_zero,
  SUM(CASE WHEN ABS(loan_percent_income - loan_amnt / person_income) > 0.01 THEN 1 ELSE 0 END) AS pct_income_mismatch_gt_1pct
FROM workspace.bronze.credit_risk_raw

In [0]:
%sql
-- Step 5 (spec input): Loan proportions — historical loan-status distribution with explicit denominator
SELECT
  loan_status,
  COUNT(*) AS n,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total,
  ROUND(AVG(loan_amnt), 0) AS avg_loan_amnt,
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY loan_amnt) AS median_loan_amnt
FROM workspace.bronze.credit_risk_raw
GROUP BY loan_status
ORDER BY loan_status

## 1. Source Data Inspection Report (verified against `workspace.bronze.credit_risk_raw`)

All numbers below come from the executed SQL cells above — nothing is assumed.

### 1.1 Schema (12 columns, 32,581 rows)

| # | Column | Type | Nulls | Null % | Observed range / values |
|---|--------|------|-------|--------|-------------------------|
| 1 | `person_age` | int | 0 | 0% | 20 – 144 (median 26, p95 40) |
| 2 | `person_income` | int | 0 | 0% | 4,000 – 6,000,000 (median 55,000) |
| 3 | `person_home_ownership` | string | 0 | 0% | RENT (16,446), MORTGAGE (13,444), OWN (2,584), OTHER (107) |
| 4 | `person_emp_length` | double | 895 | 2.7% | 0 – 123 years (median 4) |
| 5 | `loan_intent` | string | 0 | 0% | 6 values: EDUCATION, MEDICAL, VENTURE, PERSONAL, DEBTCONSOLIDATION, HOMEIMPROVEMENT |
| 6 | `loan_grade` | string | 0 | 0% | A (10,777) … G (64) — 7 values |
| 7 | `loan_amnt` | int | 0 | 0% | 500 – 35,000 (median 8,000) |
| 8 | `loan_int_rate` | double | 3,116 | 9.6% | 5.42 – 23.22 % (median 10.99, mean 11.01) |
| 9 | `loan_status` | int | 0 | 0% | 0 = 25,473 (78.18%), 1 = 7,108 (21.82%) |
| 10 | `loan_percent_income` | double | 0 | 0% | 0.00 – 0.83 (median 0.15) |
| 11 | `cb_person_default_on_file` | string | 0 | 0% | N (26,836), Y (5,745) |
| 12 | `cb_person_cred_hist_length` | int | 0 | 0% | 2 – 30 years |

### 1.2 Verified data-quality findings

| Finding | Count | Detail |
|---------|-------|--------|
| Exact duplicate rows (all 12 columns) | 165 | 32,581 rows → 32,416 distinct |
| `person_age` > 100 | 5 | Impossible ages (max observed = 144) |
| `person_age` 91–100 | 1 | Borderline, plausible but rare |
| `person_emp_length` > `person_age` | 2 | Logically impossible work history |
| `person_income` ≥ 1,000,000 | 9 | Extreme incomes (max 6M) |
| `loan_int_rate` = 0 | 0 | **No zero-rate rows — no rule needed** (instructor question answered by the data) |
| `loan_percent_income` ≠ `loan_amnt` / `person_income` (tolerance 0.01) | 388 | Column is approximately derivable but not exactly |

### 1.3 Loan proportions (explicit denominator: all 32,581 Bronze rows)

* Historical loan status: 0 (no default) = 25,473 (78.18%); 1 (default) = 7,108 (21.82%).
* Average loan amount: 9,237 (status 0) vs 10,851 (status 1); medians 8,000 vs 9,600.

## 2. Assumptions & Items Needing Clarification

### 2.1 Verified assumptions

* A1. Source CSV is at `/Volumes/workspace/bronze/landing_zone/credit_risk_dataset.csv`; Bronze table `workspace.bronze.credit_risk_raw` already exists with the 12 source columns (schema verified above).
* A2. UC schemas `workspace.bronze`, `workspace.silver`, `workspace.gold` exist; silver/gold tables from a previous run exist and will be **rebuilt** per the approved spec.
* A3. `loan_status` is the historical outcome: 1 = default, 0 = no default. It is the **label**, never an input feature.
* A4. No date/timestamp column exists in the source — no time-based filtering or partitioning is possible.
* A5. There is no unique applicant/loan ID column, so identity is only the full 12-column tuple.
* A6. **Environment note:** the currently attached compute is a Serverless SQL warehouse (SQL-only). The PySpark implementation (per the exercise) will run on serverless notebook compute instead — the spec is identical either way.

### 2.2 Open clarifications — no decision applied until answered

> The instructor's document explicitly says: *record the agreed answers, do not decide silently.* None of the defaults below are applied yet.

| # | Question | Why it matters | Candidate (NOT applied) |
|---|----------|----------------|--------------------------|
| **C-1** | **What is the instructor's exact deterministic risk-segmentation rule?** The only rule found anywhere is in the old notebook: *HIGH_RISK if `loan_percent_income` > 0.3 AND `person_home_ownership` = 'RENT', else STANDARD_RISK*. | Gold cannot be built without the approved rule. Instructor doc says "ask, do not invent". | Use the old notebook rule **only after the instructor confirms it verbatim**, incl. category labels and boundary handling |
| C-2 | Remove or only report the 165 exact duplicates? | Affects row counts in Silver and all downstream proportions | Drop exact duplicates in Silver (row count 32,581 → 32,416) |
| C-3 | `person_emp_length`: impute or drop the 895 nulls? | Imputing changes distribution; dropping loses ~2.7% of data | Impute with median (4.0) — to be confirmed |
| C-4 | `loan_int_rate`: impute or drop the 3,116 nulls (9.6%)? | Too many to drop silently; method affects grade-based analyses | Impute with global median (10.99) — to be confirmed |
| C-5 | Rule for `person_age` > 100 (5 rows). Keep the 91–100 row? | Instructor: "age > 100 → instructor-approved rule" | Drop the 5 rows with age > 100; keep the 91–100 row |
| C-6 | Rule for `person_emp_length` > `person_age` (2 rows)? | Logically impossible records | Drop the 2 rows |
| C-7 | Rule for `person_income` ≥ 1M (9 rows)? | Extreme outliers skew mean-based stats | Keep unchanged — to be confirmed |
| C-8 | Recompute `loan_percent_income` where it disagrees with `loan_amnt/person_income` (388 rows > 0.01 tolerance)? | Source column is the authority vs. internal consistency | Keep the source value as-is in Silver; report only |
| C-9 | Should Gold keep an `Unclassified` category? | Instructor doc explicitly allows it for rows where the rule can't apply or inputs are missing | Include `Unclassified` only for rows with null rule inputs (after C-3/C-4 imputation there should be 0, verified by test) |
| C-10 | Should Bronze add metadata columns (`source_file`, `ingested_at`)? | Instructor: "add simple metadata, if appropriate" | Add both columns to Bronze only — to be confirmed |

## 3. Bronze Layer Specification

**Purpose:** preserve the source data exactly as received — the immutable source of truth. No cleaning, no filtering, no type coercion beyond reading the CSV.

| Item | Specification |
|------|---------------|
| Source | `/Volumes/workspace/bronze/landing_zone/credit_risk_dataset.csv`, CSV with header |
| Target | `workspace.bronze.credit_risk_raw` (Delta) |
| Write mode | Full overwrite on each re-ingestion (idempotent re-load) |
| Schema | The 12 source columns with the types verified in §1.1 — no new business columns |
| Metadata | `source_file`, `ingested_at` — **pending C-10** |
| Transformations | **None.** Nulls, duplicates, and impossible values stay as-is |

**Acceptance criteria**

* AC-BR-1: Row count = 32,581 (data rows in the CSV).
* AC-BR-2: Exactly the 12 source columns (+2 metadata if C-10 approved) in source order.
* AC-BR-3: Min/max/null counts of every source column match §1.1 exactly (e.g., `loan_int_rate` nulls = 3,116).
* AC-BR-4: Bronze still contains the 165 duplicates and the 5 age>100 rows (raw preservation).

**Test cases**

* T-BR-1: `COUNT(*)` = 32,581.
* T-BR-2: Null counts per column equal §1.1 values.
* T-BR-3: Duplicate row count = 165 (unchanged from source).
* T-BR-4: Categorical values are exactly the enumerated sets in §1.1 (no unexpected spellings/casing).

## 4. Silver Layer Specification

**Purpose:** the trusted, analysis-ready layer — apply **only** the cleaning rules approved via C-2 … C-8, and record row accounting.

| Item | Specification |
|------|---------------|
| Source | `workspace.bronze.credit_risk_raw` |
| Target | `workspace.silver.credit_risk_cleaned` (Delta) |
| Write mode | Full overwrite (rebuildable from Bronze at any time) |

**Proposed cleaning rules (each gated on its clarification — none applied yet)**

| Rule ID | Operation | Rows affected | Gate |
|---------|-----------|---------------|------|
| R-SV-1 | Drop exact duplicates (all 12 columns identical) | 165 | C-2 |
| R-SV-2 | `person_emp_length` nulls → impute median 4.0 | 895 | C-3 |
| R-SV-3 | `loan_int_rate` nulls → impute median 10.99 | 3,116 | C-4 |
| R-SV-4 | Drop `person_age` > 100 | 5 | C-5 |
| R-SV-5 | Drop `person_emp_length` > `person_age` | 2 | C-6 |
| R-SV-6 | `person_income` ≥ 1M → keep unchanged | 9 | C-7 |
| R-SV-7 | `loan_percent_income` → keep source value, no recompute | 388 mismatches | C-8 |
| R-SV-8 | Standardize strings: trim whitespace; categorical values already conform to §1.1 sets | 0 expected | — |

**Row accounting (to be recorded as pipeline output)**

32,581 Bronze → 32,416 after dedup (C-2) → 32,411 after age rule (C-5) → 32,409 after emp/age rule (C-6) → **expected Silver ≈ 32,409 rows** (exact number depends on rule overlap; the pipeline must report each step's count).

**Acceptance criteria**

* AC-SV-1: Zero nulls in `person_emp_length` and `loan_int_rate` (post-imputation) *if* C-3/C-4 approve imputation.
* AC-SV-2: Zero exact duplicate rows.
* AC-SV-3: Max `person_age` ≤ 100; no row with `person_emp_length` > `person_age`.
* AC-SV-4: All categorical values still within the §1.1 sets.
* AC-SV-5: `loan_status` distribution preserved within the tolerance of the dropped rows (start: 78.18% / 21.82%).

**Test cases**

* T-SV-1: Null-count query returns 0 for the two imputed columns.
* T-SV-2: Distinct-row count = total row count.
* T-SV-3: `MAX(person_age) <= 100` and `SUM(emp_length > age) = 0`.
* T-SV-4: Row-accounting table reconciles: Bronze − dropped − deduped = Silver total.

## 5. Gold Layer Specification

**Purpose:** the business-facing consumption layer. Only addition: the `risk_segment` column, derived **exclusively** from the instructor's approved rule.

| Item | Specification |
|------|---------------|
| Source | `workspace.silver.credit_risk_cleaned` |
| Target | `workspace.gold.credit_risk_final` (Delta) |
| Write mode | Full overwrite (rebuildable from Silver) |
| Columns | All Silver columns + `risk_segment` — nothing else derived |

### 5.1 Risk-segmentation rule — ⚠️ BLOCKED on C-1

> **No rule will be invented.** The instructor's exercise document says: *"Create a column named `risk_segment` using the exact rule provided by the instructor. If the instructor has not provided the rule, ask for it."*

The only candidate found (in the old `Practice1_Leak_Water_Credit_Risk` notebook, **not confirmed as the instructor's**):

* `HIGH_RISK` if `loan_percent_income > 0.3` **AND** `person_home_ownership = 'RENT'`; else `STANDARD_RISK`

Before coding, the instructor must confirm: (a) the exact condition(s), (b) the category labels, (c) how many categories exist, (d) how boundary values are treated (e.g., is 0.30 exactly `HIGH_RISK` or not).

### 5.2 Unclassified handling (pending C-9)

Per the instructor's document, an `Unclassified` category is explicitly allowed for rows where the rule cannot be applied or required values are missing. After C-3/C-4 imputation there should be **zero** such rows — a test must verify this and report the count.

**Acceptance criteria**

* AC-GD-1: `risk_segment` contains only the approved labels (+ `Unclassified` if C-9 approved).
* AC-GD-2: Category counts and percentages sum to 100% of Gold rows, with the denominator stated.
* AC-GD-3: Every row's segment follows the approved rule exactly (spot-checkable with a verification query).
* AC-GD-4: Gold row count = Silver row count (no rows added or lost).

**Test cases**

* T-GD-1: `COUNT(DISTINCT risk_segment)` = number of approved categories.
* T-GD-2: Segment proportions sum to 100%.
* T-GD-3: Rule re-verification query returns 0 violating rows.
* T-GD-4: Gold count = Silver count.

## 6. Edge Cases

| # | Edge case | Layer | Planned handling (per gates above) |
|---|-----------|-------|-----------------------------------|
| E-1 | Age up to 144 (5 rows > 100) | Silver | Dropped under C-5; kept raw in Bronze |
| E-2 | `person_emp_length` = 123 years, longer than the person's age | Silver | Dropped under C-6 (2 rows) |
| E-3 | `person_income` = 6,000,000 (9 rows ≥ 1M) | Silver | Kept under C-7; flagged in EDA notes |
| E-4 | `loan_int_rate` null in 9.6% of rows | Silver | Imputed under C-4; whether to flag imputed rows per-row — **sub-question for instructor** |
| E-5 | `loan_percent_income` = 0 (tiny loan relative to income) | Gold | Falls to the else-branch of the rule; valid value, not a defect |
| E-6 | `loan_percent_income` disagrees with `loan_amnt/person_income` (388 rows) | Silver | Kept per C-8; documented |
| E-7 | `person_home_ownership` = 'OTHER' (107 rows) | Gold | Must be covered by the rule's else-branch or `Unclassified` — depends on C-1 wording |
| E-8 | Row identical to another except one null field | Silver | Not an exact duplicate under R-SV-1 — kept (no invented key) |
| E-9 | Re-running the pipeline on the same CSV | Bronze→Gold | Overwrite mode makes every layer idempotent — identical results expected |

## 7. EDA, Feature Selection & Loan Proportion Analysis Plan (Bronze/Silver stage)

**EDA deliverable (2–3 charts + tables), per the instructor's Step 3:**

1. Histogram of `loan_amnt` (median 8,000, range 500–35,000).
2. Histogram of `loan_int_rate` (5.42–23.22; 3,116 nulls excluded from the chart and stated).
3. Bar chart of `loan_status` counts with the 78.18% / 21.82% split.

**Feature-selection table (instructor Step 4) — from the verified schema only:**

| Column | Meaning | Useful for analysis? | Notes |
|--------|---------|--------------------|-------|
| `person_age` | Applicant age | Yes | 5 impossible values handled in Silver |
| `person_income` | Annual income | Yes | Right-skewed (9 rows ≥ 1M) |
| `person_home_ownership` | Housing status | Yes | 4 values; input to the candidate rule |
| `person_emp_length` | Years employed | Yes | 2.7% nulls |
| `loan_intent` | Loan purpose | Yes | 6 categories |
| `loan_grade` | Credit grade | Yes — descriptive only | Lender-assigned at decision time → leakage risk if used to *predict* `loan_status` |
| `loan_amnt` | Loan amount | Yes | Core EDA field |
| `loan_int_rate` | Interest rate | Yes | 9.6% nulls; also lender-assigned → leakage risk for prediction |
| `loan_status` | Historical default (1/0) | **Label — never an input** | 21.82% default proportion |
| `loan_percent_income` | Loan/income ratio | Yes | Candidate rule input; 388 internal mismatches |
| `cb_person_default_on_file` | Prior bureau default | Yes | Historical bureau flag → leakage risk for prediction |
| `cb_person_cred_hist_length` | Credit history years | Yes | Correlated with age (both usable descriptively) |

**Loan proportion analysis (instructor Step 5):**

* Loan-status counts and percentages (denominator: all Silver rows; re-run of the Bronze query above).
* Historical default proportion (same denominator; stated as descriptive of this dataset only).
* Loan amount average and median overall and per status.
* Number and percentage of applications per Gold `risk_segment` (denominator: all Gold rows).

## 8. Approval Gate

Per the instructor's document, implementation starts **only** after this spec is reviewed and C-1 (blocking) plus C-2 … C-10 are resolved. After approval, implementation proceeds layer by layer: Bronze ingestion → Silver cleaning → Gold segmentation → EDA charts → proportion analysis → validation tests.